<div style="background:#1a73e8;color:white;padding:20px 24px;border-radius:8px;margin-bottom:8px;">
  <h1 style="margin:0;font-size:1.8em;">🧬 ECABSD v2 — GPU Training on BM5</h1>
  <p style="margin:6px 0 0;opacity:0.9;">Equivariant Cross-Attention Binding Site Detection · CrossPPI Architecture</p>
</div>

> **Run cells top to bottom. Do not skip any cell.**  
> Each step checks its prerequisites so you cannot accidentally run out of order.

| Step | What it does |
|------|-------------|
| 1 | Verify GPU is allocated |
| 2 | Install all dependencies |
| 3 | Clone the latest code from GitHub |
| 4 | Download BM5-clean PDB data |
| 5 | Regenerate processed graphs (33-dim features) |
| 6 | Train the full model (200 epochs) |
| 7 | Evaluate on the held-out test set |
| 8 | Plot training curves |
| 9 | Save model checkpoint to Google Drive |

# Step 1 — Verify GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "❌ No GPU detected!\n"
        "Go to Runtime → Change runtime type → Hardware accelerator → GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✅ GPU: {gpu_name}  ({gpu_mem:.1f} GB VRAM)")
print(f"   CUDA {torch.version.cuda} · PyTorch {torch.__version__}")

# Step 2 — Install Dependencies

In [ ]:
print("Installing packages... (this takes ~60 seconds the first time)")
!pip install -q torch_geometric biopython pyyaml pydssp tqdm scikit-learn matplotlib seaborn

# Verify critical imports
import torch_geometric, Bio, yaml, pydssp
print(f"✅ torch_geometric {torch_geometric.__version__}")
print(f"✅ biopython {Bio.__version__}")
print("✅ All dependencies installed successfully")

# Step 3 — Clone Repository

In [ ]:
import os

REPO_URL  = "https://github.com/nayanees6607/ecabsd_temp.git"
REPO_DIR  = "/content/ecabsd_temp"

if os.path.exists(REPO_DIR):
    print("Repo already exists — pulling latest changes...")
    %cd {REPO_DIR}
    !git pull origin main
else:
    print("Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR}
    %cd {REPO_DIR}

# Verify key files exist
for f in ["train.py", "evaluate.py", "config.yaml",
          "models/ecabsd_model.py", "models/cross_attention.py",
          "scripts/prepare_db5.py"]:
    status = "✅" if os.path.exists(f) else "❌ MISSING"
    print(f"  {status}  {f}")

# Show model feature settings to confirm no dimension mismatch
import yaml as _yaml
cfg = _yaml.safe_load(open("config.yaml"))
print(f"\n📋 Config check:")
print(f"   input_dim          = {cfg['model']['input_dim']}  (must be 33)")
print(f"   edge_feature_dim   = {cfg['model']['edge_feature_dim']}  (must be 5)")
print(f"   hidden_dim         = {cfg['model']['hidden_dim']}")
print(f"   cross_attn_layers  = {cfg['model'].get('num_cross_attn_layers', '?')}")
print(f"   loss               = {cfg['training']['loss']}")

assert cfg['model']['input_dim'] == 33, "❌ input_dim must be 33!"
assert cfg['model']['edge_feature_dim'] == 5, "❌ edge_feature_dim must be 5!"
print("\n✅ Config dimensions verified — no mismatch")

# Step 4 — Download BM5-Clean PDB Data

In [ ]:
BM5_DIR = "/content/ecabsd_temp/data/BM5-clean"

if os.path.exists(BM5_DIR):
    pdb_count = len([f for f in os.listdir(BM5_DIR + "/HADDOCK-ready") if os.path.isdir(BM5_DIR + "/HADDOCK-ready/" + f)])
    print(f"✅ BM5-clean already downloaded ({pdb_count} complexes)")
else:
    print("Downloading BM5-clean from GitHub (~500 MB)...")
    !git clone https://github.com/haddocking/BM5-clean.git {BM5_DIR}
    print("✅ BM5-clean downloaded")

# Step 5 — Prepare Dataset

> **Important:** This step regenerates all `.pt` graph files with the correct **33-dim node features** and **5-dim edge features**.  
> Run this whenever the graph construction code changes. Skipping it with stale `.pt` files  
> will cause a dimension mismatch error (`mat1 and mat2 shapes cannot be multiplied`).

In [ ]:
import glob

PROCESSED_DIR = "/content/ecabsd_temp/data/db5_processed"
SPLITS_CSV    = "/content/ecabsd_temp/data/db5_splits.csv"

existing_pt = glob.glob(f"{PROCESSED_DIR}/*.pt")

if existing_pt:
    # Verify the first .pt file has the correct feature dimensions
    import torch as _torch
    sample = _torch.load(existing_pt[0], map_location="cpu")
    actual_dim = sample.x.shape[1]
    if actual_dim == 33:
        print(f"✅ {len(existing_pt)} processed files found with correct dim={actual_dim}")
        print("   Skipping re-processing (delete data/db5_processed/*.pt to force redo)")
    else:
        print(f"⚠️  Stale files found with dim={actual_dim} (need 33) — Regenerating...")
        !rm -f {PROCESSED_DIR}/*.pt {SPLITS_CSV}
        existing_pt = []

if not existing_pt:
    print("Running prepare_db5.py (5-10 minutes)...")
    %cd /content/ecabsd_temp
    !python scripts/prepare_db5.py \
        --db5-dir data/BM5-clean/HADDOCK-ready \
        --output-dir data/db5_processed \
        --threads 2
    new_pt = glob.glob(f"{PROCESSED_DIR}/*.pt")
    print(f"✅ Generated {len(new_pt)} graph files")

# Step 6 — Train the Model

Training runs for up to 200 epochs with:
- **Combined Focal + Soft-Dice loss** (directly optimises F1)
- **AdamW** with cosine-warmup LR schedule
- **Early stopping** on validation F1 (patience = 80)
- **CrossPPI-style bidirectional cross-attention** (4 layers)

Every epoch prints: `Train Loss | Train F1 | Val Loss | Val F1 | AUC-ROC | AUC-PR | LR`

In [ ]:
%cd /content/ecabsd_temp
!python train.py

# Step 7 — Evaluate on Test Set

Uses the **threshold saved in the checkpoint** (found on validation set only).  
The test set is **never touched** during training or threshold selection.

In [ ]:
%cd /content/ecabsd_temp
!python evaluate.py

# Print the saved metrics JSON for a clean summary
import json
metrics_path = "results/metrics.json"
if os.path.exists(metrics_path):
    m = json.load(open(metrics_path))
    print("\n" + "="*52)
    print("  ECABSD Test Set Results")
    print("="*52)
    targets = {
        "auc_roc":   (0.78, 0.84, 0.88),
        "auc_pr":    (0.40, 0.50, 0.60),
        "f1":        (0.55, 0.65, 0.72),
        "precision": (0.50, 0.65, 0.75),
        "recall":    (0.65, 0.75, 0.85),
        "mcc":       (0.40, 0.55, 0.65),
        "accuracy":  (0.78, 0.85, 0.90),
    }
    for key, (acc, strong, exc) in targets.items():
        val = m.get(key)
        if val is None: continue
        if val >= exc:   grade = "🏆 Excellent"
        elif val >= strong: grade = "✅ Strong"
        elif val >= acc:    grade = "🆗 Acceptable"
        else:               grade = "⚠️  Weak"
        print(f"  {key:>10s}: {val:.4f}  {grade}")
    print("="*52)

# Step 8 — Plot Training Curves

In [ ]:
import json, matplotlib.pyplot as plt

hist = json.load(open("logs/training_history.json"))
epochs   = [h["epoch"]            for h in hist]
tr_loss  = [h["train"]["loss"]    for h in hist]
val_loss = [h["val"]["loss"]      for h in hist]
tr_f1    = [h["train"]["f1"]      for h in hist]
val_f1   = [h["val"]["f1"]        for h in hist]
val_auc  = [h["val"].get("auc_roc", 0) for h in hist]

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("ECABSD v2 — Training History", fontsize=14, fontweight="bold")

axes[0].plot(epochs, tr_loss,  label="Train Loss", color="#1a73e8")
axes[0].plot(epochs, val_loss, label="Val Loss",   color="#ea4335", linestyle="--")
axes[0].set_title("Loss"); axes[0].legend(); axes[0].set_xlabel("Epoch")

axes[1].plot(epochs, tr_f1,  label="Train F1", color="#1a73e8")
axes[1].plot(epochs, val_f1, label="Val F1",   color="#ea4335", linestyle="--")
axes[1].axhline(0.72, color="#34a853", linestyle=":", label="Excellent (0.72)")
axes[1].set_title("F1 Score"); axes[1].legend(); axes[1].set_xlabel("Epoch")

axes[2].plot(epochs, val_auc, label="Val AUC-ROC", color="#8e44ad")
axes[2].axhline(0.88, color="#34a853", linestyle=":", label="Excellent (0.88)")
axes[2].set_title("AUC-ROC"); axes[2].legend(); axes[2].set_xlabel("Epoch")

plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plot saved to results/training_curves.png")

# Step 9 — Save Model to Google Drive

In [ ]:
from google.colab import drive
import shutil

drive.mount("/content/drive")

DRIVE_DIR = "/content/drive/MyDrive/ECABSD_v2"
os.makedirs(DRIVE_DIR, exist_ok=True)

files_to_save = [
    ("checkpoints/best_model.pt",       "best_model.pt"),
    ("logs/training_history.json",      "training_history.json"),
    ("results/metrics.json",            "metrics.json"),
    ("results/training_curves.png",     "training_curves.png"),
    ("results/confusion_matrix.png",    "confusion_matrix.png"),
]

for src, dst in files_to_save:
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DRIVE_DIR, dst))
        print(f"✅ Saved {dst}")
    else:
        print(f"⚠️  {src} not found — skipping")

print(f"\n🎉 All files saved to Google Drive: {DRIVE_DIR}")